# Numerical Integration and Convergence

## A test problem with a known answer

Numerical integration replaces a continuous area by a finite calculation. To determine whether an algorithm is working, we begin with a function whose exact integral is known:

$$f(x)=\frac{3}{2}(1-x^2), \qquad 0\leq x\leq1.$$

Direct integration gives

$$I=\int_0^1f(x)\,dx
=\frac{3}{2}\left[x-\frac{x^3}{3}\right]_0^1=1.$$

This exact value lets us measure the numerical error

$$E_N=|I_N-I|,$$

where $I_N$ is an estimate constructed with $N$ subintervals or samples. The central question is not just whether an estimate is close to 1, but **how rapidly it approaches 1 as computational effort increases**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def integrand(x):
    '''Test function with an exact integral of 1 on [0, 1].'''
    return 1.5*(1.0-x**2)


xlow = 0.0
xhigh = 1.0
exact_integral = 1.0
sample_counts = 10**np.arange(2, 7)  # 10^2 through 10^6

## Composite trapezoid rule

The trapezoid rule replaces the function by a straight line on each subinterval. With $x_i=a+ih$ and $h=(b-a)/N$,

$$I_N^{(T)}=h\left[\frac{1}{2}f(x_0)
+\sum_{i=1}^{N-1}f(x_i)+\frac{1}{2}f(x_N)\right].$$

There are $N$ subintervals but **$N+1$ endpoints**. Including both endpoints and all $N$ trapezoids is essential.

For a sufficiently smooth function,

$$E_N^{(T)}=O(h^2)=O(N^{-2}),$$

so the expected log–log convergence slope is again $-2$. The midpoint and trapezoid rules have the same formal order but generally different error constants and opposite error signs for a concave function such as this one.

In [ ]:
trapezoid_estimates = []

for n_intervals in sample_counts:
    x = np.linspace(xlow, xhigh, n_intervals+1)
    y = integrand(x)
    h = (xhigh-xlow)/n_intervals
    estimate = h*(0.5*y[0]+np.sum(y[1:-1])+0.5*y[-1])
    trapezoid_estimates.append(estimate)

trapezoid_estimates = np.asarray(trapezoid_estimates)
trapezoid_errors = np.abs(trapezoid_estimates-exact_integral)

for n_intervals, estimate, error in zip(
        sample_counts, trapezoid_estimates, trapezoid_errors):
    print(f"N={n_intervals:7d}: I_N={estimate:.15f}, absolute error={error:.3e}")

## Estimate and convergence plots

As in the midpoint notebook, numerical error is displayed as a difference from a known benchmark, not as an uncertainty on the integral estimate. The $N^{-2}$ reference line provides a direct visual test of the predicted convergence order.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(sample_counts, trapezoid_estimates, 'o-', label='Trapezoid estimate')
axes[0].axhline(exact_integral, color='k', linestyle='--', label='Exact integral')
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of subintervals, N')
axes[0].set_ylabel('Integral estimate')
axes[0].set_title('Composite Trapezoid Estimates')
axes[0].legend()

reference = trapezoid_errors[0]*(sample_counts/sample_counts[0])**(-2.0)
axes[1].loglog(sample_counts, trapezoid_errors, 'o-', label='Absolute error')
axes[1].loglog(sample_counts, reference, '--', label=r'Reference: $N^{-2}$')
axes[1].set_xlabel('Number of subintervals, N')
axes[1].set_ylabel(r'$|I_N-I|$')
axes[1].set_title('Trapezoid Convergence')
axes[1].legend()

slope = np.polyfit(np.log10(sample_counts), np.log10(trapezoid_errors), 1)[0]
print(f"Measured log-log slope = {slope:.4f} (expected approximately -2)")

plt.tight_layout()
plt.show()

## Interpretation

For this concave-down function, the straight-line trapezoids lie below the curve, whereas midpoint rectangles slightly overestimate it. Both errors decrease as $N^{-2}$, but the midpoint error is smaller for this particular function. Formal order describes the asymptotic rate, not the complete error at a particular $N$.